# Studi Kasus Webinar ke-6: AI untuk Klasifikasi Penyebab Deforestasi
## Attributing Direct Drivers to Deforestation Alerts with Satellite Imagery and AI

Notebook ini mendemonstrasikan alur klasifikasi citra menggunakan **CNN (Convolutional Neural Network)** untuk lima kelas: pertanian, pembangunan jalan, pertambangan, kebakaran, dan penebangan selektif.

**Penting:** Untuk dapat dijalankan tanpa dataset eksternal, notebook membuat citra sintetis dengan pola sederhana. Ini bukan citra Sentinel-1/Sentinel-2 dan modelnya tidak boleh digunakan untuk mengklasifikasikan deforestasi nyata. Aplikasi nyata memerlukan citra satelit autentik, label terverifikasi, dan validasi lapangan.


## 1. Instalasi pustaka
Jalankan jika pustaka belum tersedia.

In [ ]:
%pip install -q tensorflow scikit-learn matplotlib

## 2. Import pustaka

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import pandas as pd

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = [
    "Agriculture", "Road Development", "Mining",
    "Fire", "Selective Logging"
]
IMG_SIZE = 64
print("TensorFlow:", tf.__version__)
print("Kelas:", CLASS_NAMES)

## 3. Studi kasus dan pembuatan data sintetis

Model belajar mengenali pola sederhana yang dirancang untuk lima kategori. Data ini hanya untuk latihan alur machine learning; pola visualnya tidak mewakili kompleksitas citra satelit sesungguhnya.

In [ ]:
def make_patch(label, seed, size=64):
    rng = np.random.default_rng(seed)
    h = w = size
    img = np.zeros((h, w, 3), dtype=np.float32)
    img[:] = [rng.uniform(.12,.25), rng.uniform(.28,.48), rng.uniform(.10,.23)]
    yy, xx = np.mgrid[0:h, 0:w]

    if label == 0:  # pertanian: petak teratur
        step = int(rng.integers(8, 15))
        img[:, ::step, :] = [.60,.50,.30]
        img[::step, :, :] = [.60,.50,.30]
        for _ in range(3):
            x, y = int(rng.integers(0,48)), int(rng.integers(0,48))
            img[y:y+8, x:x+12] = [.62,.43,.25]
    elif label == 1:  # jalan: garis memanjang
        center = int(rng.integers(18,46))
        for y in range(h):
            cx = center + int(3*np.sin(y/8))
            img[y, max(0,cx-3):min(w,cx+3), :] = [.68,.61,.46]
    elif label == 2:  # pertambangan: area terbuka
        for _ in range(3):
            cx, cy = int(rng.integers(10,54)), int(rng.integers(10,54))
            rx, ry = int(rng.integers(5,14)), int(rng.integers(4,12))
            mask = ((xx-cx)/rx)**2 + ((yy-cy)/ry)**2 < 1
            img[mask] = [.68,.42,.23]
    elif label == 3:  # kebakaran: bekas terbakar
        cx, cy = int(rng.integers(20,44)), int(rng.integers(20,44))
        rx, ry = int(rng.integers(12,23)), int(rng.integers(10,20))
        dist = ((xx-cx)/rx)**2 + ((yy-cy)/ry)**2
        dist += rng.normal(0,.12,(h,w))
        img[dist < .75] = [.12,.12,.10]
        img[(dist >= .75) & (dist < 1.1)] = [.48,.27,.13]
    else:  # penebangan selektif: celah kecil tersebar
        for _ in range(int(rng.integers(8,18))):
            cx, cy = int(rng.integers(3,61)), int(rng.integers(3,61))
            r = int(rng.integers(1,4))
            mask = (xx-cx)**2 + (yy-cy)**2 <= r*r
            img[mask] = [.58,.42,.26]

    img += rng.normal(0,.06,img.shape)
    return np.clip(img,0,1).astype(np.float32)

samples_per_class = 240
X, y = [], []
for c in range(len(CLASS_NAMES)):
    for i in range(samples_per_class):
        X.append(make_patch(c, SEED + c*10000 + i))
        y.append(c)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]
print("Bentuk citra:", X.shape, "| Bentuk label:", y.shape)

## 4. Contoh citra dari setiap kelas

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for c, ax in enumerate(axes):
    ix = np.where(y == c)[0][0]
    ax.imshow(X[ix])
    ax.set_title(CLASS_NAMES[c])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Membagi data

Data dibagi menjadi train, validation, dan test. Pada penelitian nyata, pemisahan berdasarkan wilayah atau waktu juga perlu dipertimbangkan untuk menghindari kebocoran informasi spasial.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=.50, random_state=SEED, stratify=y_temp
)
print("Train:", len(X_train), "| Validation:", len(X_val), "| Test:", len(X_test))

## 6. Membangun model CNN

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(.05),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(.30),
    layers.Dense(64, activation="relu"),
    layers.Dense(len(CLASS_NAMES), activation="softmax")
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 7. Melatih model

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=12, batch_size=32,
    callbacks=[early_stop], verbose=1
)

## 8. Grafik proses pelatihan

In [ ]:
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(hist["accuracy"], label="Train")
axes[0].plot(hist["val_accuracy"], label="Validation")
axes[0].set_title("Akurasi"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(hist["loss"], label="Train")
axes[1].plot(hist["val_loss"], label="Validation")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Evaluasi pada data test
Gunakan metrik per kelas, bukan akurasi saja, untuk memahami kesalahan model.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
probs = model.predict(X_test, verbose=0)
y_pred = probs.argmax(axis=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(classification_report(
    y_test, y_pred, target_names=CLASS_NAMES, zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(9,7))
disp.plot(ax=ax, xticks_rotation=35, colorbar=False)
plt.title("Confusion Matrix — Data Test")
plt.tight_layout()
plt.show()

## 10. Prediksi satu citra

In [ ]:
sample_index = 0
image = X_test[sample_index]
true_id = int(y_test[sample_index])
p = model.predict(image[None, ...], verbose=0)[0]
pred_id = int(np.argmax(p))

plt.figure(figsize=(4,4))
plt.imshow(image)
plt.axis("off")
plt.title("Label asli: " + CLASS_NAMES[true_id])
plt.show()

print("Label asli:", CLASS_NAMES[true_id])
print("Prediksi:", CLASS_NAMES[pred_id])
print(f"Confidence: {p[pred_id]:.2%}")
display(pd.DataFrame({
    "Kelas": CLASS_NAMES,
    "Probabilitas": p
}).sort_values("Probabilitas", ascending=False).style.format({"Probabilitas":"{:.2%}"}))

## 11. Fungsi prediksi dan menyimpan model

Fungsi berikut menerima citra RGB 64 × 64 yang dinormalisasi ke rentang 0–1. Karena model hanya belajar dari citra sintetis, fungsi ini hanya untuk demonstrasi.

In [ ]:
def predict_driver(image_array):
    image_array = np.asarray(image_array, dtype=np.float32)
    if image_array.shape != (IMG_SIZE, IMG_SIZE, 3):
        raise ValueError(f"Citra harus berbentuk {(IMG_SIZE, IMG_SIZE, 3)}.")
    if image_array.max() > 1:
        image_array = image_array / 255.0
    prob = model.predict(image_array[None, ...], verbose=0)[0]
    best = int(prob.argmax())
    return {
        "prediksi_demo": CLASS_NAMES[best],
        "confidence": float(prob[best]),
        "peringatan": "Hasil model sintetis, bukan klasifikasi deforestasi nyata."
    }

print(predict_driver(X_test[0]))
model.save("cnn_deforestation_synthetic_demo.keras")
print("Model tersimpan: cnn_deforestation_synthetic_demo.keras")

## 12. Analisis dan keterbatasan

### Prinsip yang dipelajari
- CNN mempelajari fitur spasial dari citra.
- Dataset perlu memiliki label yang konsisten dan representatif.
- Data harus dipisahkan menjadi train, validation, dan test.
- Akurasi, *precision*, *recall*, F1-score, dan *confusion matrix* memberi gambaran evaluasi yang lebih lengkap.

### Keterbatasan
1. Data dalam notebook bersifat sintetis, bukan citra Sentinel-1 atau Sentinel-2.
2. Pola yang dibuat sederhana sehingga skor tinggi tidak menunjukkan kemampuan pada citra satelit nyata.
3. Citra optik dapat terhalang awan; data radar dan optik dapat saling melengkapi.
4. Gangguan alam dan aktivitas manusia dapat memiliki pola yang mirip.
5. Hasil untuk penggunaan nyata perlu diverifikasi dengan data lapangan dan ahli.

### Pengembangan lanjutan
- Menggunakan dataset citra satelit berlabel yang sah dan terdokumentasi.
- Menggabungkan kanal radar Sentinel-1 dan kanal optik Sentinel-2 bila tersedia.
- Menguji model pada wilayah dan waktu yang berbeda dari data pelatihan.
- Menampilkan prediksi pada peta berdasarkan koordinat dan tanggal alert.
- Melakukan validasi lapangan sebelum hasil digunakan untuk keputusan kebijakan atau penegakan hukum.

## Kesimpulan

Notebook ini memperlihatkan alur eksperimen klasifikasi citra berbasis CNN: pembuatan dataset, pelatihan, evaluasi, prediksi, dan penyimpanan model. Implementasi ini merupakan prototipe pembelajaran, bukan sistem operasional pemantauan deforestasi. Penerapan nyata membutuhkan citra satelit autentik, label terverifikasi, evaluasi yang ketat, dan validasi lapangan.
